In [8]:
from pathlib import Path
import pandas as pd
from sqlalchemy import and_, func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS224_APPLN_CPC,
)

ENV = "PROD"  # per il risultato finale rilanciare con "PROD"
OUTPUT_DIR = Path("outputs/step0")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

patstat = PatstatClient(env=ENV)
db = patstat.orm()

SECTOR_NAME = "Quantum Computing"

# Classi CPC "core": prefisso senza spazi
SECTOR_CPC_PREFIXES = (
    "G06N10/",      # Quantum computing (dal 2019), include /20 /40 /60 /70 /80
    "G06N99/002",   # Quantum computers (collocazione storica, pre-2019)
    "Y10S977/933",  # Spintronics or quantum computing (collezione storica)
)

# Classi CPC "adiacenti": valide solo se accompagnate da testo quantistico
ADJACENT_CPC_PREFIXES = (
    "B82Y10/00",    # Nanotecnologie per elaborazione dell'informazione
    "H10N60/",      # Dispositivi superconduttori
    "H01L39/",      # Dispositivi superconduttori (vecchia collocazione)
    "H03K19/195",   # Logica superconduttiva
)

SECTOR_KEYWORDS = (
    "quantum comput",   # computer, computing, computation
    "qubit",
    "quantum bit",
    "quantum processor",
    "quantum gate",
    "quantum circuit",
    "quantum annealing",
    "quantum error correction",
    "quantum algorithm",
    "quantum simulator",
)

# Keyword generiche: usate SOLO in AND con le classi CPC adiacenti
ADJACENT_KEYWORDS = ("quantum", "josephson")

START_YEAR = 2010   # anno di primo deposito della famiglia
END_YEAR = 2023

IPR_TYPE = "PI"  # "PI" = Patented Inventions (esclude i modelli di utilità)


def symbol_prefix_condition(symbol_column, codes):
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


def keyword_condition(terms):
    title = func.lower(func.coalesce(TLS202_APPLN_TITLE.appln_title, ""))
    abstract = func.lower(func.coalesce(TLS203_APPLN_ABSTR.appln_abstract, ""))
    return or_(*(col.like(f"%{t.lower()}%") for t in terms for col in (title, abstract)))


def sector_family_ids():
    family_id = TLS201_APPLN.docdb_family_id.label("docdb_family_id")

    def base():
        return (
            db.query(family_id)
            .select_from(TLS201_APPLN)
            .filter(TLS201_APPLN.docdb_family_id > 0)  # esclude domande senza famiglia
            .filter(TLS201_APPLN.earliest_filing_year.between(START_YEAR, END_YEAR))
            .filter(TLS201_APPLN.ipr_type == IPR_TYPE) 
        )

    def join_cpc(q):
        return q.join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)

    def join_text(q):
        return (
            q.outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
            .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        )

    # Perimetro CPC core
    by_cpc = join_cpc(base()).filter(
        symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, SECTOR_CPC_PREFIXES)
    )

    # Perimetro keyword specifiche
    by_keywords = join_text(base()).filter(keyword_condition(SECTOR_KEYWORDS))

    # CPC adiacente AND keyword generica
    by_adjacent = join_text(join_cpc(base())).filter(
        and_(
            symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, ADJACENT_CPC_PREFIXES),
            keyword_condition(ADJACENT_KEYWORDS),
        )
    )

    return by_cpc.union(by_keywords).union(by_adjacent)


sector_ids = sector_family_ids()

In [9]:
N_POS, N_NEG = 50, 50
NEG_MOD, NEG_REMAINDER = 1000, 7   # sottoinsieme deterministico di negativi (in PROD usa 1000 o più)

has_text = or_(TLS202_APPLN_TITLE.appln_title.isnot(None),
               TLS203_APPLN_ABSTR.appln_abstract.isnot(None))


def eligible_families(*filters):
    """Famiglie nel periodo con almeno un'application che ha titolo o abstract (qualsiasi lingua)."""
    return patstat.df(
        db.query(TLS201_APPLN.docdb_family_id.label("docdb_family_id"))
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(TLS201_APPLN.docdb_family_id > 0)
        .filter(TLS201_APPLN.earliest_filing_year.between(START_YEAR, END_YEAR))
        .filter(has_text)
        .filter(*filters)
        .distinct()
    )


in_sector = TLS201_APPLN.docdb_family_id.in_(sector_ids.subquery().select())

# 1. Positivi: famiglie nel perimetro con almeno un testo disponibile
pool_pos = eligible_families(in_sector)
df_pos = pool_pos.sample(n=min(N_POS, len(pool_pos)), random_state=42).copy()
df_pos["query_label"] = 1

# 2. Negativi: famiglie fuori dal perimetro con almeno un testo disponibile
pool_neg = eligible_families(
    ~in_sector,
    TLS201_APPLN.docdb_family_id % NEG_MOD == NEG_REMAINDER,
)
df_neg = pool_neg.sample(n=min(N_NEG, len(pool_neg)), random_state=42).copy()
df_neg["query_label"] = 0

df_validation = (
    pd.concat([df_pos, df_neg])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
family_ids_list = df_validation["docdb_family_id"].tolist()

# 3. Testi di tutte le application delle famiglie campionate (che hanno titolo o abstract)
text_query = (
    db.query(
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_id,
        func.max(TLS202_APPLN_TITLE.appln_title).label("appln_title"),
        func.max(TLS203_APPLN_ABSTR.appln_abstract).label("appln_abstract"),
    )
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(family_ids_list))
    .filter(has_text)
    .group_by(TLS201_APPLN.docdb_family_id, TLS201_APPLN.appln_id)
)
df_texts = patstat.df(text_query)

# Una application per famiglia: preferisci quella con abstract, poi con titolo, poi appln_id minore
df_texts["has_abs"] = df_texts["appln_abstract"].notna()
df_texts["has_title"] = df_texts["appln_title"].notna()
df_texts = (
    df_texts.sort_values(["docdb_family_id", "has_abs", "has_title", "appln_id"],
                         ascending=[True, False, False, True])
    .drop_duplicates("docdb_family_id")
    .drop(columns=["has_abs", "has_title"])
)

df_validation = df_validation.merge(df_texts, on="docdb_family_id", how="left")
df_validation["manual_label"] = ""

# CSV completo (con query_label) e CSV in cieco per l'etichettatura
validation_csv_path = OUTPUT_DIR / "validation_sample.csv"
df_validation.to_csv(validation_csv_path, index=False)

print(f"Positivi: {len(df_pos)} | Negativi: {len(df_neg)} | Totale: {len(df_validation)}")
print(f"CSV completo: {validation_csv_path}")
display(df_validation.head(5))

Positivi: 50 | Negativi: 50 | Totale: 100
CSV completo: outputs/step0/validation_sample.csv


,docdb_family_id,query_label,appln_id,appln_title,appln_abstract,manual_label
0,47227007,0,378713089,Packaging end cap,None,
1,64948007,0,505882855,A lithium ion battery have a high safety elect...,The invention discloses a lithium ion battery ...,
2,49636007,0,412905238,TRANSMISSION FOR ELECTRIC VEHICLE,PURPOSE: A transmission for electric vehicle i...,
3,72757816,1,539278563,Modular quantum computer software architecture...,The invention discloses a modular quantum comp...,
4,78286658,1,560140652,HIGH-TEMPERATURE SUPERCONDUCTING HYDRIDE AND M...,FIELD: conductors.SUBSTANCE: present invention...,


In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df_lab = pd.read_csv(validation_csv_path)   # dopo aver compilato manual_label con 0/1
#y_true = df_lab["manual_label"].astype(int)
y_true = [0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0]
y_pred = df_lab["query_label"].astype(int)

print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"F1:        {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(confusion_matrix(y_true, y_pred))

Accuracy:  0.9400
Precision: 0.9000
Recall:    0.9783
F1:        0.9375
[[49  5]
 [ 1 45]]
